In [53]:
import duckdb

In [54]:
%load_ext sql
%sql duckdb://
%sql INSTALL excel; LOAD excel;

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


Running query in 'duckdb://'

Success


In [55]:
%sql ROLLBACK;

Running query in 'duckdb://'

Success


In [56]:
%%sql
-- Report 1: Open Requests Log Summary
WITH customer_care AS (
    SELECT
        REGEXP_REPLACE(TRIM("Region"), '[\r\n]+', ' ', 'g') AS region,
        REGEXP_REPLACE(TRIM("Zone"), '[\r\n]+', ' ', 'g') AS zone,
        REGEXP_REPLACE(TRIM("Section"), '[\r\n]+', ' ', 'g') AS section,
        REGEXP_REPLACE(TRIM("Block/Village"), '[\r\n]+', ' ', 'g') AS village,
        "RHOA STATUS" AS rhoa_status,
        CAST("Date of Submission" AS DOUBLE) AS submission_serial,
        CAST(
                    DATE '1899-12-30'
                    + CAST(FLOOR(submission_serial) AS INTEGER)
                    AS TIMESTAMP
                )
                + (
                    submission_serial - FLOOR(submission_serial)
                ) * INTERVAL '1 day' AS submission_datetime
    FROM read_xlsx(
        'Customer Care.xlsx',
        header = true,
        all_varchar = true
    )
)
SELECT
    region,
    zone,
    section,
    village,
    COUNT(*) AS total_requests,
    SUM(
        CASE
            WHEN rhoa_status IS NULL
              OR TRIM(rhoa_status) = ''
              OR TRIM(rhoa_status) <> 'Converted'
            THEN 1
            ELSE 0
        END
    ) AS pending,
    DATE_DIFF(
            'day',
            CAST(submission_datetime AS DATE),
            CURRENT_DATE
        ) AS days_open
FROM customer_care
GROUP BY region, zone, section, village, days_open
ORDER BY pending DESC;

Running query in 'duckdb://'

region,zone,section,village,total_requests,pending,days_open
KABRAS A,CENTRAL ZONE-D,CENTRAL-D 1,SHIVIKHWA,12,12,2
KABRAS B,EASTERN ZONE,EAST 1,SOYI,9,9,3
KABRAS B,KAIMOSI,KAIMOSI 3,MUKWO,9,9,10
KABRAS B,KHAYEGA,KHAYEGA-3,SHIVEYE,8,8,1
KABRAS A,WESTERN ZONE-C,WEST-C 2,CHEGULO-A,8,8,11
KABRAS B,SOUTHERN ZONE-A,SOUTH-A 4,SIRIGOI,6,6,1
KABRAS B,SOUTHERN ZONE-A,SOUTH-A 4,MATOYI,5,5,0
KABRAS B,SOUTHERN ZONE-A,SOUTH-A 4,SIRIGOI,5,5,0
KABRAS B,SOUTHERN ZONE-A,SOUTH-A 4,MATOYI,5,5,10
KABRAS B,SOUTHERN ZONE-A,SOUTH-A 4,SIRIGOI,4,4,14


In [57]:
%%sql
-- Report 1: Open Requests Log Detailed
WITH customer_care AS (
    SELECT
        *,
        CAST("Date of Submission" AS DOUBLE) AS submission_serial
    FROM read_xlsx(
        'Customer Care.xlsx',
        header = true,
        all_varchar = true
    )
),
cleaned AS (
    SELECT
        *,
        CAST(
            DATE '1899-12-30'
            + CAST(FLOOR(submission_serial) AS INTEGER)
            AS TIMESTAMP
        )
        + (
            submission_serial - FLOOR(submission_serial)
        ) * INTERVAL '1 day' AS submission_datetime
    FROM customer_care
)
SELECT
    "Response Id" AS response_id,
    submission_datetime AS date_of_submission,
    "Region" AS region,
    "Zone" AS zone,
    "Section" AS section,
    "Famer Name." AS farmer_name,
    "Farmer Phone No." AS farmer_phone,
    "Requests" AS request_type,
    COALESCE(
        NULLIF(TRIM("Request Details"), ''),
        'Not Started'
    ) AS request_details,
    COALESCE(
        NULLIF(TRIM("RHOA STATUS"), ''),
        'Not Started'
    ) AS rhoa_status,
    COALESCE(
        NULLIF(TRIM("REMARKS AT CUSTOMER CARE"), ''),
        'Not Started'
    ) AS remarks_customer_care,
    DATE_DIFF(
        'day',
        CAST(submission_datetime AS DATE),
        CURRENT_DATE
    ) AS days_open
FROM cleaned
WHERE "RHOA STATUS" IS NULL
   OR TRIM("RHOA STATUS") = ''
   OR "RHOA STATUS" IN ('Not Started', 'In Progress')
ORDER BY days_open DESC;

Running query in 'duckdb://'

response_id,date_of_submission,region,zone,section,farmer_name,farmer_phone,request_type,request_details,rhoa_status,remarks_customer_care,days_open
bb0d29f4,2026-09-03 12:19:16,KABRAS B,SOUTHERN ZONE-A,SOUTH-A 4,Paul Amanza kikin,7.1657758E8,Permit Mill Cane,Not Started,Not Started,Not Started,16
163e41ad,2026-09-03 15:59:49,KABRAS B,KAIMOSI,KAIMOSI 3,Hellen kitur,7.28148585E8,Permit Mill Cane,Not Started,Not Started,Not Started,16
fbc3bbc4,2026-09-03 12:23:32,BUTERE,SHIANDA ZONE,SHIANDA-3,Josephat Opworah,7.25703894E8,Permit Mill Cane,Not Started,Not Started,Not Started,16
1a873300,2026-09-03 12:25:22,BUTERE,SHIANDA ZONE,SHIANDA-2,Faith Mwaro,7.26795699E8,Permit Mill Cane,Not Started,Not Started,Not Started,16
7c6b73a0,2026-09-03 12:28:16,KABRAS B,EASTERN ZONE,EAST 5,Symon khayega,7.11325354E8,Fertilizer,Not Started,Not Started,Not Started,16
43feab17,2026-09-03 12:29:57,KHALABA,NAMULUNGU,NAMULUNGU 1,GEORGE OMULAMA NETIA,7.17607895E8,Permit Seed Cane,Not Started,Not Started,Not Started,16
f98a3028,2026-09-03 12:30:38,KABRAS B,SHINYALU,SHINYALU-1,Aggrey Shujaa lilumbi,7.25825433E8,Seed Cane,Not Started,Not Started,Not Started,16
c375ddb1,2026-09-03 12:35:24,KABRAS A,WESTERN ZONE-D,WEST-D 1,Jotham kundu Matala,7.15753083E8,Fertilizer,Not Started,Not Started,Not Started,16
29936c2b,2026-09-03 12:39:02,KABRAS A,WESTERN ZONE-A,WEST-A 5,Milcah Osimbo Liwa,7.1072972E8,Filtermud (Only Appoved Areas),Not Started,Not Started,Not Started,16
5.4191728E7,2026-09-03 12:40:06,KHALABA,BULONDO,BULONDO-1,Boaz Juma Mayende,7.9916514E8,Permit Mill Cane,Not Started,Not Started,Not Started,16


In [58]:
%%sql
-- Report 2: Resolved Requests Log Summary
WITH customer_care AS (
    SELECT
        REGEXP_REPLACE(TRIM("Region"), '[\r\n]+', ' ', 'g') AS region,
        REGEXP_REPLACE(TRIM("Zone"), '[\r\n]+', ' ', 'g') AS zone,
        REGEXP_REPLACE(TRIM("Section"), '[\r\n]+', ' ', 'g') AS section,
        REGEXP_REPLACE(TRIM("Block/Village"), '[\r\n]+', ' ', 'g') AS village,
        "RHOA STATUS" AS rhoa_status
    FROM read_xlsx(
        'Customer Care.xlsx',
        header = true,
        all_varchar = true
    )
)
SELECT
    region,
    zone,
    section,
    village,
    COUNT(*) AS total_requests,
    SUM(CASE WHEN TRIM(rhoa_status) = 'Converted' THEN 1 ELSE 0 END) AS resolved
FROM customer_care
GROUP BY region, zone, section, village
ORDER BY resolved DESC;

Running query in 'duckdb://'

region,zone,section,village,total_requests,resolved
KABRAS B,EASTERN ZONE,EAST 5,IVAKALE-A,3,0
MAGUT,MAGUT,MAGUT 2,NG-ENYI-A,2,0
KABRAS B,SOUTHERN ZONE-A,SOUTH-A 3,MUNENGA,3,0
KABRAS A,WESTERN ZONE-C,WEST-C 4,CHENJENI,4,0
KABRAS A,WESTERN ZONE-A,WEST-A 5,MUSIDI,3,0
KABRAS B,CENTRAL ZONE-B,CENTRAL-B 4,BUKHAKUNGA,3,0
KABRAS B,SOUTHERN ZONE-B,SOUTH-B 3,KWAVA,3,0
MAGUT,MAGUT,MAGUT 4,MAJENGO,4,0
KABRAS B,KAIMOSI,KAIMOSI 3,TAUNET A,2,0
KABRAS B,KAIMOSI,KAIMOSI 3,MUSANYI,2,0


In [59]:
%%sql
-- Report 2: Resolved Requests Log Detailed
WITH customer_care AS (
    SELECT
        *,
        CAST("Date of Submission" AS DOUBLE) AS submission_serial
    FROM read_xlsx(
        'Customer Care.xlsx',
        header = true,
        all_varchar = true
    )
),
cleaned AS (
    SELECT
        *,
        CAST(
            DATE '1899-12-30'
            + CAST(FLOOR(submission_serial) AS INTEGER)
            AS TIMESTAMP
        )
        + (
            submission_serial - FLOOR(submission_serial)
        ) * INTERVAL '1 day' AS submission_datetime
    FROM customer_care
)
SELECT
    "Response Id" AS response_id,
    submission_datetime AS date_of_submission,
    "Region" AS region,
    "Zone" AS zone,
    "Section" AS section,   
    "Block/Village" AS village,
    "Famer Name." AS farmer_name,
    "Requests" AS request_type,
    COALESCE(
        NULLIF(TRIM("CLOSED THROUGH FIELD NUMBER"), ''),
        'Not Started'
    ) AS closed_through_field_number,
    COALESCE(
        NULLIF(TRIM("RHOA REMARKS"), ''),
        'Not Started'
    ) AS rhoa_remarks,
    COALESCE(
            NULLIF(TRIM("REMARKS AT CUSTOMER CARE"), ''),
            'Not Started'
        ) AS remarks_customer_care
FROM cleaned
WHERE TRIM("RHOA STATUS") = 'Converted'
ORDER BY date_of_submission DESC;

Running query in 'duckdb://'

response_id,date_of_submission,region,zone,section,village,farmer_name,request_type,closed_through_field_number,rhoa_remarks,remarks_customer_care


In [60]:
%%sql
-- Report 3: Farmer Request History Summary

WITH customer_care AS (
    SELECT
        "Farmer Id" AS farmer_id,
        REGEXP_REPLACE(TRIM("Famer Name."), '[\r\n]+', ' ', 'g') AS farmer_name,
        TRIM("Farmer Phone No.") AS farmer_phone_no,
        REGEXP_REPLACE(TRIM("Region"), '[\r\n]+', ' ', 'g') AS region,
        REGEXP_REPLACE(TRIM("Zone"), '[\r\n]+', ' ', 'g') AS zone,
        REGEXP_REPLACE(TRIM("Section"), '[\r\n]+', ' ', 'g') AS section,
        REGEXP_REPLACE(TRIM("Block/Village"), '[\r\n]+', ' ', 'g') AS village,
        TRIM("Requests") AS requests,
        CAST("Date of Submission" AS DOUBLE) AS submission_serial
    FROM read_xlsx(
        'Customer Care.xlsx',
        header = true,
        all_varchar = true
    )
),

cleaned AS (
    SELECT
        *,
        CAST(
            DATE '1899-12-30'
            + CAST(FLOOR(submission_serial) AS INTEGER)
            AS TIMESTAMP
        )
        + (
            submission_serial - FLOOR(submission_serial)
        ) * INTERVAL '1 day' AS submission_datetime
    FROM customer_care
)

SELECT
    farmer_id,
    farmer_name,
    region,
    zone,
    section,
    village,
    COUNT(*) AS total_requests,
    STRING_AGG(requests, ', ') AS request_types,
    MAX(submission_datetime) AS most_recent_request
FROM cleaned
GROUP BY
    farmer_id,
    farmer_name,
    region,
    zone,
    section,
    village
ORDER BY total_requests DESC;

Running query in 'duckdb://'

farmer_id,farmer_name,region,zone,section,village,total_requests,request_types,most_recent_request
1.0541323E7,Mary Afandi,MAGUT,MAGUT,MAGUT 4,MAJENGO,3,"Permit Seed Cane, Permit Seed Cane, Permit Seed Cane",2026-09-15 12:22:28
1.4555586E7,Joseph Ukokhe Waswa,BUTERE,SIMAKINA,SIMAKINA 1,SISOKHE,3,"Contracting (Registration), Contracting (Registration), Contracting (Registration)",2026-09-04 16:05:36
3.3798818E7,Celestine Jepkemboi,SELIA,KAIBOI,KAIBOI 1,KAOBOI CENTRAL,3,"Seed Cane, Seed Cane, Seed Cane",2026-09-12 12:07:17
566316.0,Joseph mwera,KABRAS A,CENTRAL ZONE-D,CENTRAL-D 1,SHIVIKHWA,3,"Seed Cane, Seed Cane, Seed Cane",2026-09-17 16:47:46
1.1303247E7,Andrew chiboli,KABRAS A,CENTRAL ZONE-D,CENTRAL-D 1,SHIVIKHWA,3,"Fertilizer, Fertilizer, Fertilizer",2026-09-17 16:58:53
6996166.0,Shadrack M. Molonje,KABRAS A,CENTRAL ZONE-D,CENTRAL-D 3,KIMANG-ETI,3,"Permit Mill Cane, Contracting (Registration), Permit Mill Cane",2026-09-05 10:53:13
2.7172686E7,Jemima mwachi,MAGUT,PANPAPER,PANPAPER 1,MARULA,3,"Seed Cane, Seed Cane, Seed Cane",2026-09-15 08:43:59
3.3776163E7,Brian mtamba,KABRAS B,SOUTHERN ZONE-A,SOUTH-A 3,EMULAMA,3,"Permit Mill Cane, Permit Mill Cane, Permit Mill Cane",2026-09-14 08:36:50
2.9532618E7,Peter Munyanya Baraza,BUTERE,MALAHA,MALAHA 1,KHABONDI,3,"Permit Mill Cane, Permit Mill Cane, Permit Mill Cane",2026-09-14 16:48:54
2.5639957E7,Eline Busienei,KABRAS A,CENTRAL ZONE-C,CENTRAL-C 1,TUMBENI,2,"Filtermud (Only Appoved Areas), Seed Cane",2026-09-05 12:08:24


In [61]:
%%sql
-- Report 3: Farmer Request History Detailed

WITH customer_care AS (
    SELECT
        "Farmer Id" AS farmer_id,
        REGEXP_REPLACE(TRIM("Famer Name."), '[\r\n]+', ' ', 'g') AS farmer_name,
        TRIM("Farmer Phone No.") AS farmer_phone_no,
        REGEXP_REPLACE(TRIM("Region"), '[\r\n]+', ' ', 'g') AS region,
        REGEXP_REPLACE(TRIM("Zone"), '[\r\n]+', ' ', 'g') AS zone,
        REGEXP_REPLACE(TRIM("Section"), '[\r\n]+', ' ', 'g') AS section,
        REGEXP_REPLACE(TRIM("Block/Village"), '[\r\n]+', ' ', 'g') AS village,
        TRIM("Requests") AS request,
        CAST("Date of Submission" AS DOUBLE) AS submission_serial
    FROM read_xlsx(
        'Customer Care.xlsx',
        header = true,
        all_varchar = true
    )
),

cleaned AS (
    SELECT
        *,
        CAST(
            DATE '1899-12-30'
            + CAST(FLOOR(submission_serial) AS INTEGER)
            AS TIMESTAMP
        )
        + (
            submission_serial - FLOOR(submission_serial)
        ) * INTERVAL '1 day' AS submission_datetime
    FROM customer_care
)

SELECT
    farmer_id,
    farmer_name,
    farmer_phone_no,
    region,
    zone,
    section,
    village,
    request,
    MAX(submission_datetime) AS most_recent_request
FROM cleaned
GROUP BY
    farmer_id,
    farmer_name,
    farmer_phone_no,
    region,
    zone,
    section,
    village,
    request
ORDER BY most_recent_request DESC;

Running query in 'duckdb://'

farmer_id,farmer_name,farmer_phone_no,region,zone,section,village,request,most_recent_request
3.6731901E7,BENSON WANJALA,7.99566615E8,KABRAS B,CENTRAL ZONE-B,CENTRAL-B 3,SHAMBERERE,Permit Mill Cane,2026-09-19 11:13:06
2.543786E7,DINAH MUSUTSA,7.2643814E8,KABRAS A,CENTRAL ZONE-A,CENTRAL-A 2,MWERA,Permit Mill Cane,2026-09-19 11:12:43
2.2029609E7,Ezekiel Kakai,7.34586221E8,KABRAS B,EASTERN ZONE,EAST 2,BULOVI,Permit Mill Cane,2026-09-19 11:09:35
2.09644E7,Ndetswa Maclean,7.25670343E8,KABRAS A,CENTRAL ZONE-D,CENTRAL-D 1,SHIVIKHWA,Filtermud (Only Appoved Areas),2026-09-19 11:08:01
6626075.0,HENRY AMUKAYIA,7.19158686E8,KABRAS B,SHINYALU,SHINYALU-4,SHIRULU,Permit Mill Cane,2026-09-19 11:03:43
3.6952603E7,KEVIN PHILIP,1.00492424E8,KABRAS B,SOUTHERN ZONE-A,SOUTH-A 3,EMULAMA,Permit Mill Cane,2026-09-19 11:00:50
1.3584167E7,charles burudi,7.26572256E8,KABRAS B,SOUTHERN ZONE-B,SOUTH-B 1,CHEVOSO,Filtermud (Only Appoved Areas),2026-09-19 10:58:47
5120125.0,LAWRENCE MASINDE MATSUKHU,7.20005545E8,KABRAS B,MAKUNGA,MAKUNGA-5,SHINOYI,Permit Mill Cane,2026-09-19 10:57:52
3456196.0,Samuel Muyeka,7.14526492E8,KABRAS B,SOUTHERN ZONE-A,SOUTH-A 2,EMACHEMBE,Permit Mill Cane,2026-09-19 10:50:05
6314441.0,ELPHAS WAMBUTSU,7.51430887E8,KABRAS B,MAKUNGA,MAKUNGA 3,MUTONO,Fertilizer,2026-09-19 10:50:05
